In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, sosfilt, stft as scipy_stft
import timm

# ── Paths ─────────────────────────────────────────────────────────────────────
COMP_DIR = '/kaggle/input/hms-harmful-brain-activity-classification'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

# ── Автопошук найновішого .ckpt по модальності та версії ──────────────────────
# Очікувана структура датасету на Kaggle:
#   eeg_v2_2026-05-07/best.ckpt   ← завантаж папку цілком
# Або просто: eeg_best.ckpt       ← якщо один файл

def find_latest_ckpt(modality):
    """
    Шукає .ckpt файли де в шляху є {modality}_vN або назва файлу містить modality.
    Повертає той що з найбільшим N.
    """
    candidates = []
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if not f.endswith('.ckpt'):
                continue
            full = os.path.join(root, f)
            path_lower = full.lower()

            # шукаємо {modality}_vN в будь-якій частині шляху
            version = None
            for part in full.split(os.sep):
                if part.lower().startswith(f'{modality}_v'):
                    n_str = part[len(modality) + 2:].split('_')[0]
                    if n_str.isdigit():
                        version = int(n_str)

            # якщо версію не знайшли — перевіряємо чи modality є хоч десь в шляху
            if version is None:
                if modality in path_lower:
                    version = 0
                else:
                    continue  # цей файл не для нашої модальності

            candidates.append((version, full))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[0], reverse=True)
    best_v, best_path = candidates[0]
    print(f'  [{modality}] знайдено v{best_v}: {best_path}')
    return best_path


print('=== Всі .ckpt файли у /kaggle/input ===')
found_any = False
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.ckpt'):
            print(f'  {os.path.join(root, f)}')
            found_any = True
if not found_any:
    print('  (не знайдено — перевір що датасет з вагами додано як Input)')

print()
EEG_CKPT  = find_latest_ckpt('eeg')
SPEC_CKPT = find_latest_ckpt('spec')

print(f'\nEEG_CKPT  = {EEG_CKPT}')
print(f'SPEC_CKPT = {SPEC_CKPT}')

device: cpu
=== Всі .ckpt файли у /kaggle/input ===
  (не знайдено — перевір що датасет з вагами додано як Input)


EEG_CKPT  = None
SPEC_CKPT = None


In [2]:
# ── EEG & Spec preprocessing constants (повинні збігатись з src/dataset.py) ──

VOTE_COLS    = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
EEG_ALL_COLS = ['Fp1','F3','C3','P3','F7','T3','T5','O1','Fz','Cz',
                'Pz','Fp2','F4','C4','P4','F8','T4','T6','O2','EKG']
SPEC_GROUPS  = ['LL', 'RL', 'LP', 'RP']
SPEC_TIMES   = 300   # time steps to extract per sample (each = 2 sec)

EEG_FS             = 200
EEG_WINDOW_SAMPLES = 10_000
EEG_N_FFT          = 128
EEG_HOP            = 50

_LL = ['Fp1','F7','T3','T5','O1']
_RL = ['Fp2','F8','T4','T6','O2']
_LP = ['Fp1','F3','C3','P3','O1']
_RP = ['Fp2','F4','C4','P4','O2']
_CHAINS = [_LL, _RL, _LP, _RP]

_BUTTER_SOS  = butter(N=5, Wn=[0.5, 20.0], btype='bandpass', fs=EEG_FS, output='sos')
_STFT_FREQS  = np.fft.rfftfreq(EEG_N_FFT, d=1.0 / EEG_FS)
_FREQ_MASK   = _STFT_FREQS <= 20.0
EEG_FREQ_BINS  = int(_FREQ_MASK.sum())                              # 13
EEG_TIME_STEPS = (EEG_WINDOW_SAMPLES - EEG_N_FFT) // EEG_HOP + 1  # 198

In [ ]:
# ── Datasets ──────────────────────────────────────────────────────────────────

class HMSEEGDataset(Dataset):
    def __init__(self, df, eeg_dir):
        self.df      = df.reset_index(drop=True)
        self.eeg_dir = eeg_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        eeg_id = int(row['eeg_id'])
        offset = float(row.get('eeg_label_offset_seconds', 0))

        raw  = pd.read_parquet(f'{self.eeg_dir}/{eeg_id}.parquet')
        data = raw[EEG_ALL_COLS].values.astype(np.float32)
        col_idx = {c: i for i, c in enumerate(EEG_ALL_COLS)}

        start = int(offset * EEG_FS)
        end   = start + EEG_WINDOW_SAMPLES
        if end > len(data):
            end   = len(data)
            start = max(0, end - EEG_WINDOW_SAMPLES)
        segment = data[start:end]
        if len(segment) < EEG_WINDOW_SAMPLES:
            pad     = np.zeros((EEG_WINDOW_SAMPLES - len(segment), segment.shape[1]), dtype=np.float32)
            segment = np.concatenate([segment, pad], axis=0)

        leads = []
        for chain in _CHAINS:
            for a, b in zip(chain[:-1], chain[1:]):
                leads.append(segment[:, col_idx[a]] - segment[:, col_idx[b]])
        eeg = np.stack(leads, axis=0)  # (16, 10000)

        for i in range(eeg.shape[0]):
            ch = eeg[i]
            mask = np.isnan(ch)
            if mask.any():
                ch[mask] = ch[~mask].mean() if (~mask).any() else 0.0
        np.clip(eeg, -1024.0, 1024.0, out=eeg)
        eeg  = sosfilt(_BUTTER_SOS, eeg, axis=1).astype(np.float32)
        mean = eeg.mean(axis=1, keepdims=True)
        std  = eeg.std(axis=1, keepdims=True) + 1e-6
        eeg  = (eeg - mean) / std

        specs = []
        for ch in range(eeg.shape[0]):
            _, _, Zxx = scipy_stft(eeg[ch], fs=EEG_FS, nperseg=EEG_N_FFT,
                                   noverlap=EEG_N_FFT - EEG_HOP, boundary=None, padded=False)
            specs.append(np.abs(Zxx)[_FREQ_MASK, :EEG_TIME_STEPS])
        spec = np.log1p(np.stack(specs, axis=0)).astype(np.float32)  # (16, 13, 198)

        return torch.from_numpy(spec), torch.zeros(6)


class HMSSpectrogramDataset(Dataset):
    def __init__(self, df, spec_dir):
        self.df       = df.reset_index(drop=True)
        self.spec_dir = spec_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        spec_id = int(row['spectrogram_id'])
        raw     = pd.read_parquet(f'{self.spec_dir}/{spec_id}.parquet')

        arrays = []
        for g in SPEC_GROUPS:
            cols = [c for c in raw.columns if c.startswith(f'{g}_')]
            arrays.append(raw[cols].values.T.astype(np.float32))
        spec = np.stack(arrays, axis=0)  # (4, 100, T)

        # each time step = 2 sec; extract SPEC_TIMES steps starting at label offset
        offset_sec = float(row.get('spectrogram_label_offset_seconds', 0))
        start = int(offset_sec / 2)
        t = spec.shape[2]
        start = min(start, max(0, t - SPEC_TIMES))
        end = start + SPEC_TIMES
        if end <= t:
            spec = spec[:, :, start:end]
        else:
            chunk = spec[:, :, start:t]
            pad = np.zeros((spec.shape[0], spec.shape[1], SPEC_TIMES - chunk.shape[2]), dtype=np.float32)
            spec = np.concatenate([chunk, pad], axis=2)
        np.nan_to_num(spec, copy=False)

        return torch.from_numpy(spec), torch.zeros(6)

In [ ]:
# ── Моделі (повинні збігатись з src/eeg_model.py & src/spec_model.py) ─────────
from tqdm.notebook import tqdm

class EEGModel(nn.Module):
    def __init__(self, num_classes=6, drop_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=False, in_chans=1,
            num_classes=0, global_pool='',
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Dropout(drop_rate), nn.Linear(1280, num_classes))

    def forward(self, x):
        B, C, F, T = x.shape               # (B, 16, 13, 198)
        x = x.reshape(B, C * F, T)         # (B, 208, 198)
        x = x.unsqueeze(1)                 # (B, 1, 208, 198)
        return self.head(self.pool(self.backbone(x)).flatten(1))


class SpectrogramModel(nn.Module):
    def __init__(self, num_classes=6, drop_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=False, in_chans=4,
            num_classes=0, global_pool='avg',
        )
        self.head = nn.Sequential(nn.Dropout(drop_rate), nn.Linear(1280, num_classes))

    def forward(self, x):
        return self.head(self.backbone(x))


def load_model(model, ckpt_path):
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=True)
    state = {k.replace('model.', '', 1): v for k, v in ckpt['state_dict'].items()}
    model.load_state_dict(state)
    return model.to(device).eval()


@torch.no_grad()
def predict(model, loader, desc='inference'):
    probs = []
    for x, _ in tqdm(loader, desc=desc, unit='batch'):
        probs.append(F.softmax(model(x.to(device)), dim=1).cpu().numpy())
    return np.concatenate(probs, axis=0)

In [ ]:
# ── Test data ─────────────────────────────────────────────────────────────────

test_df = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))
sub     = pd.read_csv(os.path.join(COMP_DIR, 'sample_submission.csv'))

test_df['eeg_id'] = test_df['eeg_id'].astype(int)
if 'eeg_label_offset_seconds' not in test_df.columns:
    test_df['eeg_label_offset_seconds'] = 0.0
if 'spectrogram_label_offset_seconds' not in test_df.columns:
    test_df['spectrogram_label_offset_seconds'] = 0.0

print(f'test samples : {len(test_df)}')
print(f'sub template : {sub.columns.tolist()}')
print(test_df)

In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────────

all_probs = []

if EEG_CKPT and os.path.exists(EEG_CKPT):
    print(f'EEG  ← {EEG_CKPT}')
    ds     = HMSEEGDataset(test_df, os.path.join(COMP_DIR, 'test_eegs'))
    loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2)
    probs  = predict(load_model(EEGModel(), EEG_CKPT), loader, desc='EEG')
    all_probs.append(probs)
    print(f'  shape={probs.shape}  row_sums={probs.sum(axis=1).round(4)}')
else:
    print(f'EEG checkpoint не знайдено: {EEG_CKPT}')

if SPEC_CKPT and os.path.exists(SPEC_CKPT):
    print(f'Spec ← {SPEC_CKPT}')
    ds     = HMSSpectrogramDataset(test_df, os.path.join(COMP_DIR, 'test_spectrograms'))
    loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2)
    probs  = predict(load_model(SpectrogramModel(), SPEC_CKPT), loader, desc='Spec')
    all_probs.append(probs)
    print(f'  shape={probs.shape}  row_sums={probs.sum(axis=1).round(4)}')
else:
    print(f'Spec checkpoint не знайдено або SPEC_CKPT=None — пропущено')

if not all_probs:
    raise RuntimeError('Жоден чекпоінт не знайдено. Перевір MODEL_DIR і EEG_CKPT.')

final_probs = np.mean(all_probs, axis=0)
print(f'\nEnsemble: {len(all_probs)} model(s)  shape={final_probs.shape}')

In [ ]:
# ── Submission ────────────────────────────────────────────────────────────────
# Заповнюємо sample_submission.csv — гарантує правильний формат для Kaggle

sub[VOTE_COLS] = final_probs

sub.to_csv('submission.csv', index=False)

print('Saved: submission.csv')
print(sub)